# Search for magnetic induction strictly inside 35 km of Eros

This notebook uses only measurements with spacecraft center distance `r < 35 km`. The boundary is strict and is applied to every original sample before time binning. It searches for a spatially repeatable magnetic response without importing the wake geometry or polarization expectations of earlier work.

The primary physical template is the external field of a dipole induced by the slowly varying ambient field, proportional to $(8.4\,\mathrm{km}/r)^3[3\hat r\hat r^T-I]B_{ext}$. A constant permanent dipole is fitted as a nuisance control. Evidence for induction requires out-of-block predictive improvement over that control, stability across bin and background windows, a position-shift result stronger than chance, a constrained response coefficient, and compatible lag and radial behavior.

In [ ]:
from csv import DictWriter
from datetime import datetime
from pathlib import Path
import sys
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
repository_directory = Path("/Users/danywaller/code/dear-near")
if str(repository_directory) not in sys.path:
    sys.path.insert(0, str(repository_directory))

from scripts.near_induction_methods import (
    block_bootstrap_response,
    block_sufficient_statistics,
    fit_induction_models,
    model_arrays,
    position_shift_test,
    prepare_close_bins,
    scan_lags,
    solve_statistics,
)
from scripts.near_wave_discovery_methods import read_full_resolution_nso

In [ ]:
data_directory = Path("/Users/danywaller/Projects/near/data")
output_directory = Path("/Users/danywaller/Projects/near/output/induction_below_35km")
configuration_catalog = output_directory / "configuration_scan.csv"
primary_statistics = output_directory / "primary_induction_test.npz"
injection_statistics = output_directory / "injection_recovery.npz"
output_directory.mkdir(parents=True, exist_ok=True)

start = "2000-02-14T15:33:00"
stop = "2001-02-11T00:00:00"
maximum_distance_km = 35.0
reference_radius_km = 8.4
bin_seconds_values = np.array([10, 30, 60])
background_seconds_values = np.array([1800, 3600, 10800])
primary_bin_seconds = 30
primary_background_seconds = 3600
block_gap_seconds = 1800
minimum_bins_per_block = 20
position_shift_draws = 250
bootstrap_draws = 1000
random_seed = 433

base_settings = {
    "maximum_distance_km": maximum_distance_km,
    "block_gap_seconds": block_gap_seconds,
    "minimum_bins_per_block": minimum_bins_per_block,
    "btotal_absolute_tolerance_nt": 0.25,
    "btotal_relative_tolerance": 0.10,
}

## Apply the strict spatial selection to the full-resolution archive

The original samples are loaded at their archived cadence. Field-vector magnitude is checked against the reported total field before selection. Robust time-bin medians prevent 1 s intervals from receiving ten times the weight of 10 s intervals, but no sample at or beyond 35 km can enter a retained bin. Gaps longer than 30 minutes define independent close-pass blocks.

In [ ]:
times, values = read_full_resolution_nso(data_directory, start, stop)
raw_distance = np.linalg.norm(values[:, 5:8], axis=1)
raw_close = raw_distance < maximum_distance_km
print(f"loaded {times.size:,} full-resolution post-insertion samples")
print(f"strictly inside 35 km: {np.count_nonzero(raw_close):,} samples")
print(f"close-range span: {times[raw_close].min()} to {times[raw_close].max()}")
print(f"distance range: {raw_distance[raw_close].min():.3f} to {raw_distance[raw_close].max():.6f} km")
assert np.all(raw_distance[raw_close] < maximum_distance_km)

In [ ]:
raw_close_times = times[raw_close].astype("datetime64[ms]").astype(datetime)
figure, axes = plt.subplots(2, 1, figsize=(12, 6), constrained_layout=True)
axes[0].scatter(raw_close_times[::50], raw_distance[raw_close][::50], s=2, alpha=0.4)
axes[0].axhline(maximum_distance_km, color="black", linewidth=0.8)
axes[0].set_ylabel("center distance (km)")
axes[0].set_title("Availability of strictly selected close-range data")
axes[0].xaxis.set_major_formatter(mdates.ConciseDateFormatter(axes[0].xaxis.get_major_locator()))
axes[1].hist(raw_distance[raw_close], bins=np.linspace(19, 35, 65), color="tab:blue")
axes[1].set_xlabel("center distance (km)")
axes[1].set_ylabel("full-resolution samples")
figure.savefig(output_directory / "close_data_coverage.png", dpi=180)
figure

## Sensitivity across time bins and sliding background windows

For each configuration, a centered sliding mean estimates the slowly varying ambient field separately inside each close pass. The remaining field is fitted with three nested spatial models: a permanent dipole only, that nuisance model plus one isotropic induced-response coefficient, and the nuisance model plus a general 3 by 3 response tensor. Reported $R^2$ values are leave-one-close-pass-out predictions, so a more flexible response is penalized when it does not repeat in unseen passes. Distance, time, and orbit labels are never used as outcome weights.

In [ ]:
datasets = {}
configuration_results = []
for bin_seconds in bin_seconds_values:
    settings = dict(base_settings)
    settings["bin_seconds"] = int(bin_seconds)
    data = prepare_close_bins(times, values, settings)
    datasets[int(bin_seconds)] = data
    assert np.all(data["distance_km"] < maximum_distance_km)
    for background_seconds in background_seconds_values:
        result = fit_induction_models(
            data, int(background_seconds), reference_radius_km
        )
        configuration_results.append(result)

with configuration_catalog.open("w", newline="") as stream:
    columns = list(configuration_results[0])
    writer = DictWriter(stream, fieldnames=columns)
    writer.writeheader()
    for result in configuration_results:
        writer.writerow({key: value.tolist() if isinstance(value, np.ndarray) else value for key, value in result.items()})

print(" bin background  bins blocks  permanent_r2  scalar_delta  tensor_delta  response")
for result in configuration_results:
    print(
        f"{result['bin_seconds']:4.0f} {result['background_seconds'] / 60:9.0f}m "
        f"{result['bin_count']:6,d} {result['block_count']:6,d} "
        f"{result['permanent_cv_r2']:13.6f} {result['scalar_delta_cv_r2']:13.6f} "
        f"{result['tensor_delta_cv_r2']:13.6f} {result['scalar_response']:9.4f}"
    )
print(f"saved {configuration_catalog}")

In [ ]:
scalar_delta = np.array([result["scalar_delta_cv_r2"] for result in configuration_results]).reshape(len(bin_seconds_values), -1)
tensor_delta = np.array([result["tensor_delta_cv_r2"] for result in configuration_results]).reshape(len(bin_seconds_values), -1)
response = np.array([result["scalar_response"] for result in configuration_results]).reshape(len(bin_seconds_values), -1)
figure, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for axis, matrix, title, label, cmap in zip(
    axes,
    [scalar_delta, tensor_delta, response],
    ["isotropic induction", "anisotropic induction", "isotropic response coefficient"],
    [r"$\Delta R^2_{cv}$", r"$\Delta R^2_{cv}$", r"$\alpha$"],
    ["coolwarm", "coolwarm", "PiYG"],
):
    limit = np.nanmax(np.abs(matrix))
    image = axis.imshow(matrix, aspect="auto", cmap=cmap, vmin=-limit, vmax=limit)
    axis.set_xticks(np.arange(len(background_seconds_values)), [f"{value / 60:g}" for value in background_seconds_values])
    axis.set_yticks(np.arange(len(bin_seconds_values)), [f"{value:g}" for value in bin_seconds_values])
    axis.set_xlabel("background window (minutes)")
    axis.set_ylabel("robust bin (seconds)")
    axis.set_title(title)
    figure.colorbar(image, ax=axis, label=label)
figure.savefig(output_directory / "configuration_sensitivity.png", dpi=180)
figure

## Predeclared primary test

The primary configuration is fixed above at 30 s bins and a 60 minute sliding background, rather than selected from the sensitivity grid. Circularly shifting spacecraft positions independently within each close pass preserves magnetic autocorrelation and the distribution of sampled positions while breaking their alignment. The one-sided p-value asks whether adding the induced response improves held-out prediction more than shifted geometries do. A close-pass block bootstrap estimates uncertainty in the response coefficient.

In [ ]:
generator = np.random.default_rng(random_seed)
primary_data = datasets[primary_bin_seconds]
primary_result = fit_induction_models(
    primary_data, primary_background_seconds, reference_radius_km
)
shift_result = position_shift_test(
    primary_data,
    primary_background_seconds,
    reference_radius_km,
    position_shift_draws,
    generator,
)
bootstrap_response = block_bootstrap_response(
    primary_data,
    primary_background_seconds,
    reference_radius_km,
    bootstrap_draws,
    generator,
)
response_interval = np.percentile(bootstrap_response, [2.5, 97.5])
np.savez_compressed(
    primary_statistics,
    **primary_result,
    position_shift_null=shift_result["null"],
    position_shift_pvalue=shift_result["pvalue"],
    bootstrap_response=bootstrap_response,
)
print(f"primary scalar response: {primary_result['scalar_response']:.5f}")
print(f"block-bootstrap 95% interval: [{response_interval[0]:.5f}, {response_interval[1]:.5f}]")
print(f"observed added predictive R2: {shift_result['observed_delta_cv_r2']:.7f}")
print(f"shifted 95% interval: [{shift_result['null_lower']:.7f}, {shift_result['null_upper']:.7f}]")
print(f"one-sided position-shift p: {shift_result['pvalue']:.4f}")

figure, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].hist(shift_result["null"], bins=30, color="0.7", edgecolor="white")
axes[0].axvline(shift_result["observed_delta_cv_r2"], color="tab:red", label="observed")
axes[0].set_xlabel(r"induction $\Delta R^2_{cv}$")
axes[0].set_ylabel("position shifts")
axes[0].legend()
axes[1].hist(bootstrap_response, bins=35, color="tab:blue", alpha=0.75)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_xlabel(r"isotropic response $\alpha$")
axes[1].set_ylabel("block bootstrap draws")
figure.savefig(output_directory / "primary_placebo_and_bootstrap.png", dpi=180)
figure

## Synthetic injection and recovery on the real sampling

Known isotropic dipole responses are added to the actual measured field using its sampled positions and primary ambient-field proxy, after which the complete background estimation and blocked fit are repeated. This exposes attenuation caused by deriving the driver from the local magnetometer. With this normalization, an ideal perfectly diamagnetic sphere would approach a response of `-0.5`; failure to recover injections near that scale means the data and proxy cannot place a useful physical upper limit even when the blind test is null.

In [ ]:
injected_response = np.array([-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0])
primary_arrays = model_arrays(
    primary_data, primary_background_seconds, reference_radius_km
)
injection_template = primary_arrays["scalar_design"][:, :, -1]
recovered_response = np.empty(injected_response.size)
recovered_delta = np.empty(injected_response.size)
for index, injection in enumerate(injected_response):
    synthetic_data = dict(primary_data)
    synthetic_data["field_nt"] = primary_data["field_nt"] + injection * injection_template
    result = fit_induction_models(
        synthetic_data, primary_background_seconds, reference_radius_km
    )
    recovered_response[index] = result["scalar_response"]
    recovered_delta[index] = result["scalar_delta_cv_r2"]
np.savez_compressed(
    injection_statistics,
    injected_response=injected_response,
    recovered_response=recovered_response,
    recovered_delta_cv_r2=recovered_delta,
)
print("injected  recovered  added predictive R2")
for injected, recovered, delta in zip(injected_response, recovered_response, recovered_delta):
    print(f"{injected:8.2f} {recovered:10.4f} {delta:20.7f}")

figure, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(injected_response, recovered_response, marker="o")
axes[0].plot(injected_response, primary_result["scalar_response"] + injected_response, color="0.5", linestyle="--", label="ideal recovery")
axes[0].axvline(-0.5, color="tab:red", linewidth=0.8, label="ideal diamagnetic sphere")
axes[0].set_xlabel("injected response")
axes[0].set_ylabel("recovered response")
axes[0].legend()
axes[1].plot(injected_response, recovered_delta, marker="o")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].axvline(-0.5, color="tab:red", linewidth=0.8)
axes[1].set_xlabel("injected response")
axes[1].set_ylabel(r"induction $\Delta R^2_{cv}$")
figure.savefig(output_directory / "injection_recovery.png", dpi=180)
figure

## Lag and radial diagnostics

A conducting response to a changing ambient field may lag rather than track instantaneously. The lag scan is descriptive because the local low-pass field is only a proxy for the unmeasured upstream IMF. The radial diagnostic projects the field remaining after the fitted permanent dipole onto the fitted induced-template direction and compares it with the model's $r^{-3}$ expectation. Neither diagnostic can rescue a failed predictive/placebo test.

In [ ]:
lag_seconds = np.arange(-1800, 1801, 300)
lag_delta = scan_lags(
    primary_data,
    primary_background_seconds,
    reference_radius_km,
    lag_seconds,
)
arrays = model_arrays(
    primary_data, primary_background_seconds, reference_radius_km
)
statistics = block_sufficient_statistics(
    arrays["residual_nt"], arrays["scalar_design"], primary_data["block"]
)
coefficients = solve_statistics(statistics)
permanent_prediction = np.einsum("nij,j->ni", arrays["permanent_design"], coefficients[:3])
induced_vector = arrays["scalar_design"][:, :, -1]
induced_amplitude = np.linalg.norm(induced_vector, axis=1)
induced_unit = induced_vector / np.maximum(induced_amplitude[:, None], np.finfo(float).tiny)
observed_projection = np.sum((arrays["residual_nt"] - permanent_prediction) * induced_unit, axis=1)
expected_projection = coefficients[-1] * induced_amplitude
distance_edges = np.array([19, 22, 25, 28, 31, 33, 35])
distance_centers = (distance_edges[:-1] + distance_edges[1:]) / 2
observed_median = np.full(distance_centers.size, np.nan)
expected_median = np.full(distance_centers.size, np.nan)
for index, (lower, upper) in enumerate(zip(distance_edges[:-1], distance_edges[1:])):
    selected = (primary_data["distance_km"] >= lower) & (primary_data["distance_km"] < upper)
    if np.any(selected):
        observed_median[index] = np.median(observed_projection[selected])
        expected_median[index] = np.median(expected_projection[selected])

figure, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(lag_seconds / 60, lag_delta, marker="o")
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_xlabel("ambient-field lag (minutes)")
axes[0].set_ylabel(r"induction $\Delta R^2_{cv}$")
axes[0].set_title("Lag sensitivity")
axes[1].plot(distance_centers, observed_median, marker="o", label="observed projection")
axes[1].plot(distance_centers, expected_median, marker="s", label=r"fitted $r^{-3}$ response")
axes[1].set_xlabel("center distance (km)")
axes[1].set_ylabel("median projected residual (nT)")
axes[1].set_title("Radial consistency")
axes[1].legend()
figure.savefig(output_directory / "lag_and_radial_diagnostics.png", dpi=180)
figure

## Compare with induction physics only after the data tests

For a uniform conductor, the electromagnetic skin depth is $\delta=\sqrt{2/(\mu_0\sigma\omega)}$. The table below shows whether the tested 30-180 minute variations would penetrate a substantial fraction of an 8.4 km reference radius for illustrative conductivities. This is context, not an inversion: Eros is irregular, likely heterogeneous, and the magnetic-only single-spacecraft data do not independently measure the upstream driving field.

In [ ]:
mu0 = 4 * np.pi * 1e-7
conductivity_values = np.array([1e-6, 1e-4, 1e-2])
period_seconds_values = background_seconds_values
print("skin depth in km")
print("sigma_Sm  " + "  ".join(f"{period / 60:6.0f} min" for period in period_seconds_values))
for conductivity in conductivity_values:
    angular_frequency = 2 * np.pi / period_seconds_values
    skin_depth_km = np.sqrt(2 / (mu0 * conductivity * angular_frequency)) / 1000
    print(f"{conductivity:8.1e} " + " ".join(f"{value:10.2f}" for value in skin_depth_km))

## Interpretation and future analysis

A nonzero fitted coefficient alone is not a detection because slowly varying IMF structure and orbit phase can bias an in-sample regression. The primary evidence is the added leave-one-pass-out prediction together with the position-shift p-value. Consistent positive improvement across neighboring bin/background configurations, a bootstrap interval excluding zero, a physically plausible lag, and an increasing radial projection toward Eros would strengthen the case. Failure of these checks should be reported as no detected induction signature at the tested scales, not as proof that Eros cannot support induction.

The currently missing observable is a simultaneous upstream magnetic field. Future analysis could time-shift external solar-wind monitor to Eros and replace the local low-pass proxy, then fit a complex frequency-dependent response with the same blocked validation and spatial placebos.